# Graph Neural Networks


Figures are from Prince, *[Understanding Deep Learning](https://udlbook.github.io/udlbook/)*,
unless another source is cited. Code cells adapted from the UDL notebooks (MIT
license,
[github.com/udlbook/udlbook/tree/main/Notebooks](https://github.com/udlbook/udlbook/tree/main/Notebooks)).


## Graph Neural Networks

Neural architectures that process graphs.

Three challenges:

1. Variable topology
2. Size (billions of nodes)
3. Single monolithic graph

## Topics

- Basic definition and examples
- Graph representation
- Properties of Adjacency Matrix
- Graph neural network, tasks and loss functions
- Graph convolutional network
- Graph & Node classification
- Edge graphs

# Basic definition and examples

We start with what a graph is and with the many kinds of data that are naturally
described as graphs.


## Graph (Network)

- general structure composed of ***nodes*** (vertices) and ***edges*** (links)
- edges can be ***undirected*** or ***directed***
- a graph with directed edges and no cycles (no loops) is called ***directed acyclic graph*** (DAG)


![](figs/20-Graph-Neural-Networks/graph-nodes-edges.png)


## Directed Example – Feed Forward Network

![A fully connected network is a directed graph.](figs/20-Graph-Neural-Networks/feedforward-network-dag.png)

## Directed Example – Bayesian Graphical Model

![A latent variable model as a directed graph. Figure: Rocca, "Understanding Variational Autoencoders (VAEs)", 2019.](figs/20-Graph-Neural-Networks/bayesian-graphical-model.png)

From lecture 17.

## Undirected Examples

![](figs/20-Graph-Neural-Networks/undirected-graph-examples.png)

**road networks**<br>
**nodes**: physical locations or landmarks<br>
**edges**: connecting roads


**chemical molecules**<br>
**nodes**: atoms<br>
**edges**: chemical bonds


**electrical circuits**<br>
**nodes**: components or junctions<br>
**edges**: wires/electrical connections


## Examples

![](figs/20-Graph-Neural-Networks/social-network-graph.png)

**social networks**<br>
**nodes**: people<br>
**edges**: friendships<br>
(undirected)


![](figs/20-Graph-Neural-Networks/citation-graph.png)

**science literature**<br>
**nodes**: papers<br>
**edges**: citations<br>
(acyclic directed)


![](figs/20-Graph-Neural-Networks/knowledge-graph.png)

**knowledge graph**<br>
**nodes**: objects<br>
**edges**: named relationship<br>
(cyclic directed)


## Example – Geometric Point Cloud

![Source: <https://en.wikipedia.org/wiki/Vertex_(computer_graphics)>](figs/20-Graph-Neural-Networks/point-cloud-graph.png)

**nodes**: positions in 3D space (vertex in 3D graphics)<br>
**edges**: connections to nearby points<br>
(undirected)

## Example – Scene Graph

![](figs/20-Graph-Neural-Networks/scene-graph.png)

hierarchical graph showing relationship between objects in a 3D scene

**nodes**: composite graphs or objects in 3D space<br>
**edges**: relationships between objects, e.g. a part belongs to an object, an object to the room

Fernandez-Madrigal and Gonzalez, "Multi-hierarchical graph search," 2002<br>
Armeni et al, "3D Scene Graph: A structure for unified semantics, 3D space and camera," 2019<br>
Wald et al, "Learning 3D Semantic Scene Graphs with Instance Embeddings," 2022


## Other examples

- Wikipedia – nodes are articles, edges are hyperlinks between articles
- Computer programs – nodes are syntax tokens, edges are computation between tokens (tensor graph from lecture 05)
- Protein interactions – nodes are proteins, edges exist where two proteins interface
- Set or list – every element is connected to every other element
- image – each pixel is a node with edges to the eight adjacent pixels

# Graph representation

To process a graph with a neural network we need to store it as numbers: an
adjacency matrix for the structure, plus matrices of node and edge embeddings.


## Graph representation


![](figs/20-Graph-Neural-Networks/graph-edge-embeddings.png)


Example undirected graph with 6 nodes

Information about a node is stored in a ***node embedding***


Information about an edge is stored in an ***edge embedding***


## Graph representation – adjacency matrix

![](figs/20-Graph-Neural-Networks/graph-adjacency-matrix.png)


Assume we have $N$ nodes

The graph connections can be represented by an ***adjacency matrix***

Where a value of 1 at $(m, n)$ represents a connection between nodes $m$ and $n$.

For undirected graphs the matrix is always symmetric about the diagonal

Diagonal is zero – no edge to itself

Can be very sparse


## Graph representation – node data matrix


![](figs/20-Graph-Neural-Networks/node-data-matrix.png)

All the node data in the form of node embeddings can be represented by a ***Node data matrix***

Where $D$ is the dimension of the node embedding and

$N$ is the number of nodes


## Graph representation – edge data matrix


![](figs/20-Graph-Neural-Networks/edge-data-matrix.png)

Similarly, all the edge embedding information can be stored in an ***Edge data matrix***, where:

$D_E$ is the dimension of the edge embedding vector and

$E$ is the number of edges



# Properties of Adjacency Matrix

The adjacency matrix is more than a lookup table: multiplying by it moves
information along the edges of the graph, which is what graph neural network
layers will exploit.


## Adjacency Matrix

![](figs/20-Graph-Neural-Networks/adjacency-example-graph.png)

Assume we have an 8-node undirected graph


$$
\mathbf{A} = \begin{bmatrix}
0 & 1 & 0 & 1 & 0 & 0 & 0 & 0 \\
1 & 0 & 1 & 1 & 1 & 0 & 0 & 0 \\
0 & 1 & 0 & 0 & 1 & 0 & 0 & 0 \\
1 & 1 & 0 & 0 & 1 & 0 & 0 & 0 \\
0 & 1 & 1 & 1 & 0 & 1 & 0 & 1 \\
0 & 0 & 0 & 0 & 1 & 0 & 1 & 1 \\
0 & 0 & 0 & 0 & 0 & 1 & 0 & 0 \\
0 & 0 & 0 & 0 & 1 & 1 & 0 & 0
\end{bmatrix}
$$

Adjacency matrix for this graph.


$$
\mathbf{x} = \begin{bmatrix} 0 & 0 & 0 & 0 & 0 & 1 & 0 & 0 \end{bmatrix}^T
$$

We can one hot encode representation of node 6


## Adjacency Matrix: multiplying by $\mathbf{A}$

If we pre-multiply the one-hot encoded data node vector $\mathbf{x}$ by adjacency
matrix $\mathbf{A}$ we get the 6^th^ column of $\mathbf{A}$ indicating direct
connections to other nodes

$$
\mathbf{A}\mathbf{x} = \begin{bmatrix} 0 & 0 & 0 & 0 & 1 & 0 & 1 & 1 \end{bmatrix}^T
$$

![One-hot encoding vector of all nodes directly connected node 6.](figs/20-Graph-Neural-Networks/adjacency-ax-graph.png)


If we pre-multiply again by $\mathbf{A}$, we get a vector showing the number of
times we can get to each node in 2 steps.

$$
\mathbf{A}^2\mathbf{x} = \begin{bmatrix} 0 & 1 & 1 & 1 & 1 & 3 & 0 & 1 \end{bmatrix}^T
$$

![Graph showing all nodes that can be reached in *exactly* 2 steps.](figs/20-Graph-Neural-Networks/adjacency-a2x-graph.png)


## Adjacency Matrix: $\mathbf{A}^2$

Pre-multiplying $\mathbf{x}$ by $\mathbf{A}$ twice is equivalent to the matrix $\mathbf{A}^2$

Shows how many times you can get from node $m$ to node $n$ in 2 steps

$$
\mathbf{A}^2 = \begin{bmatrix}
2 & 1 & 1 & 1 & 2 & 0 & 0 & 0 \\
1 & 4 & 1 & 2 & 2 & 1 & 0 & 1 \\
1 & 1 & 2 & 2 & 1 & 1 & 0 & 1 \\
1 & 2 & 2 & 3 & 1 & 1 & 0 & 1 \\
2 & 2 & 1 & 1 & 5 & 1 & 1 & 1 \\
0 & 1 & 1 & 1 & 1 & 3 & 0 & 1 \\
0 & 0 & 0 & 0 & 1 & 0 & 1 & 1 \\
0 & 1 & 1 & 1 & 1 & 1 & 1 & 2
\end{bmatrix}
$$

## Adjacency Matrix: powers of $\mathbf{A}$

![](figs/20-Graph-Neural-Networks/adjacency-example-graph.png)

Example for $L = 2$: the matrix $\mathbf{A}^2$ above.


When you raise the adjacency matrix to the power of $L$ (pre-multiply L-1 times),

the entry at position $(m, n)$ of $\mathbf{A}^L$ contains the number of unique
walks of length $L$ from node $n$ to node $m$

[Note]{.underline}: this is not the same as the number of unique paths since it
includes routes that visit the same node more than once.

a non-zero entry at position $(m, n)$ indicates that the distance from $m$ to $n$
must be less than or equal to $L$.

*See UDL notebook 13.1 – Encoding Graphs*


The numbers on the last three slides, computed. The cell builds $\mathbf{A}$
from the edge list of the eight-node graph, multiplies the one-hot vector for
node 6 by it once and twice, and forms $\mathbf{A}^2$. Its diagonal counts the
walks of length 2 that leave a node and come straight back, one per edge, so it
equals the node degrees. Powers of $\mathbf{A}$ also give walk counts of any
length, and the shortest-path distance between two nodes: the first $L$ at
which their entry of $\mathbf{A}^L$ becomes non-zero. Change `node` or `edges`
and rerun.

In [ ]:
# Adapted from UDL notebook 13.1 (MIT)
import numpy as np

edges = [(1, 2), (1, 4), (2, 3), (2, 4), (2, 5), (3, 5), (4, 5),
         (5, 6), (5, 8), (6, 7), (6, 8)]          # nodes numbered from 1
N = 8
node = 6                                           # the one-hot node

A = np.zeros((N, N), dtype=int)
for m, n in edges:
    A[m - 1, n - 1] = A[n - 1, m - 1] = 1          # undirected: symmetric

x = np.zeros(N, dtype=int)
x[node - 1] = 1
A2 = A @ A

print("A symmetric:", np.array_equal(A, A.T))
print("A x   =", A @ x)
print("A^2 x =", A2 @ x)
print("A^2 =\n", A2)
print("diag(A^2)    =", np.diag(A2))
print("node degrees =", A.sum(axis=0))

def walks(m, n, L):
    return np.linalg.matrix_power(A, L)[m - 1, n - 1]

def distance(m, n):
    AL = np.eye(N, dtype=int)
    for L in range(N):
        if AL[m - 1, n - 1] > 0:
            return L
        AL = AL @ A
    return None                                    # not connected

print("walks of length 3 from node 4 to node 8:", walks(4, 8, 3))
print("shortest distance from node 1 to node 7:", distance(1, 7))

## Permutation of node indices

Since node indexing is arbitrary, we can permute the node indices

![](figs/20-Graph-Neural-Networks/permutation-original-graph.png)


$$
\begin{array}{c}
\begin{matrix} (1 & \;2 & \;\;3 & \;\;4) \end{matrix} \\
\mathbf{X} = \begin{bmatrix} 1 & 2 & 3 & 4 \\ 5 & 6 & 7 & 8 \\ 9 & 10 & 11 & 12 \end{bmatrix}
\end{array}
$$

node data


$$
\mathbf{A} = \begin{bmatrix} 0 & 1 & 0 & 0 \\ 1 & 0 & 1 & 1 \\ 0 & 1 & 0 & 1 \\ 0 & 1 & 1 & 0 \end{bmatrix}
$$

adjacency matrix


## Permutation of node indices (cont.)

$$
\begin{array}{c}
\begin{matrix} (3 & 4 & 2 & 1) \end{matrix} \\
\mathbf{P} = \begin{bmatrix} 0 & 0 & 0 & 1 \\ 0 & 0 & 1 & 0 \\ 1 & 0 & 0 & 0 \\ 0 & 1 & 0 & 0 \end{bmatrix}
\end{array}
$$

We can express this mathematically with a permutation matrix, $\mathbf{P}$


New: (1 2 3 4)<br>
Old: (3 4 2 1)

$$
\mathbf{X}' = \mathbf{X}\mathbf{P} = \begin{bmatrix} 3 & 4 & 2 & 1 \\ 7 & 8 & 6 & 5 \\ 11 & 12 & 10 & 9 \end{bmatrix}
$$

Permute the columns of the Node data matrix


$$
\mathbf{A}' = \mathbf{P}^{\mathbf{T}}\mathbf{A}\mathbf{P} = \begin{bmatrix} 0 & 1 & 1 & 0 \\ 1 & 0 & 1 & 0 \\ 1 & 1 & 0 & 1 \\ 0 & 0 & 1 & 0 \end{bmatrix}
$$

Permute both the rows and column of the Adjacency matrix


![The relabelled graph.](figs/20-Graph-Neural-Networks/permutation-permuted-graph.png)

# Graph neural network, tasks and loss functions

With a graph encoded as $\mathbf{X}$ and $\mathbf{A}$, a graph neural network
transforms the node embeddings layer by layer. We first look at what the network
is asked to predict, and at the output heads and losses for each kind of task.


## Graph Neural Network

- A graph neural network is a model that takes the node embeddings $\mathbf{X}$ and
  the adjacency matrix $\mathbf{A}$ as inputs and passes them through a series of
  $K$ layers.
- The node embeddings are updated at each layer to create intermediate "hidden"
  representations $\mathbf{H}_k$ ($k < K$) before finally computing output embeddings $\mathbf{H}_K$.
- At the start of this network, each column of the input node embeddings $\mathbf{X}$
  just contains information about the node itself.
- At the end, each column of the model output $\mathbf{H}_K$ includes information
  about the node and its context within the graph.
- This is like word embeddings passing through a transformer network. These
  represent words at the start but represent the word meanings in the context of
  the sentence at the end.

## Graph Level Tasks

Determine

- class categories, e.g. molecule is poisonous
- regression values, e.g. molecule boiling and freezing point

based on graph structure and node embeddings

For graph-level tasks, the output node embeddings are combined (e.g., by
averaging), and the resulting vector is mapped via a linear transformation or
neural network to a fixed-size vector

## Typical Three Types of Models

- Graph level regression & classification
- Node level regression & classification
- Edge prediction

*Look at prediction heads first.*


## Graph level regression & classification

![The figure shows a three-class classifier; the formulas below are for a single output (regression or binary classification).](figs/20-Graph-Neural-Networks/graph-level-task.png)

**Last layer (Regression):**

$$
y = \beta_K + \omega_K \mathbf{H}_K \mathbf{1} \,/N
$$

**Last layer (Classification):**

$$
\mathrm{Pr}(y = 1 \mid \mathbf{X}, \mathbf{A}) = \mathrm{sigmoid}\big[\beta_K + \omega_K \underbrace{\mathbf{H}_K \mathbf{1}}_{\text{Mean pooling}} /N\big]
$$


$\beta_K$ is scalar<br>
$\omega_K$ is $1 \times D$ row vector<br>
$\mathbf{H}_K$ is the $D \times N$ output embedding matrix<br>
$\mathbf{1}$ is an $N \times 1$ column vector of 1s

**Regression Loss Function:** Least Squares Loss<br>
**Classification Loss Function:** (Binary) Cross Entropy


## Node level binary regression & classification

![](figs/20-Graph-Neural-Networks/node-level-task.png)

**Last layer (Regression):**

$$
y^{(n)} = \beta_K + \omega_K \mathbf{h}_K^{(n)}
$$

**Last layer (Classification):**

$$
\mathrm{Pr}\big(y^{(n)} = 1 \mid \mathbf{X}, \mathbf{A}\big) = \mathrm{sigmoid}\big[\beta_K + \omega_K \mathbf{h}_K^{(n)}\big]
$$


$\mathbf{h}_K^{(n)}$ is the $D \times 1$ output embedding vector node for $n$

**Regression Loss Function:** Least Squares Loss<br>
**Classification Loss Function:** (Binary) Cross Entropy


## Edge prediction (classification)

Predict whether edge should exist or not.

![](figs/20-Graph-Neural-Networks/edge-prediction-task.png)

**Last layer:**

$$
\mathrm{Pr}\big(y^{(mn)} = 1 \mid \mathbf{X}, \mathbf{A}\big) = \mathrm{sigmoid}\big[\underset{[1 \times D]}{\mathbf{h}_K^{(m)T}} \, \underset{[D \times 1]}{\mathbf{h}_K^{(n)}}\big]
$$

**Classification Loss Function:** Binary Cross Entropy

# Graph convolutional network

The body of the network is a stack of layers that each update every node's
embedding from its neighbors. Graph convolutional networks do this with shared
parameters and the adjacency matrix.


## Graph convolutional network

These models are convolutional in that they update each node by aggregating
information from nearby nodes.

As such, they induce a relational inductive bias (i.e., a bias toward
prioritizing information from neighbors).

$$
\begin{aligned}
\mathbf{H}_1 &= \mathbf{F}[\mathbf{X}, \mathbf{A}, \phi_0] \\
\mathbf{H}_2 &= \mathbf{F}[\mathbf{H}_1, \mathbf{A}, \phi_1] \\
\mathbf{H}_3 &= \mathbf{F}[\mathbf{H}_2, \mathbf{A}, \phi_2] \\
\vdots \;&=\; \vdots \\
\mathbf{H}_K &= \mathbf{F}[\mathbf{H}_{K-1}, \mathbf{A}, \phi_{K-1}],
\end{aligned}
$$


A function $F[\cdot]$ with parameters $\phi_i$ that takes the node embeddings and
adjacency matrix and outputs new node embeddings


## Equivariance and Invariance

Every layer should be ***equivariant*** to index permutations

$$
\mathbf{H}_{k+1}\mathbf{P} = \mathbf{F}[\mathbf{H}_k\mathbf{P}, \mathbf{P}^T\mathbf{A}\mathbf{P}, \phi_k]
$$

And for graph-level tasks the output should be ***invariant*** to index
permutations; node- and edge-level outputs are equivariant (UDL §13.5)

$$
y = \mathrm{sigmoid}[\beta_K + \omega_K \mathbf{H}_K \mathbf{1}/N\,] = \mathrm{sigmoid}[\beta_K + \omega_K \mathbf{H}_K \mathbf{P}\mathbf{1}/N\,]
$$

## Example Graph Convolution Network (GCN) layer

At each node $n$ in layer $k$, aggregate information from neighboring nodes

$$
\mathrm{agg}[n, k] = \sum_{m \in \mathrm{ne}[n]} \mathbf{h}_k^{(m)}
$$

where $\mathrm{ne}[n]$ returns the set of indices of the neighbors of node $n$.

![](figs/20-Graph-Neural-Networks/gcn-aggregation-neighbors.png)


$$
\mathrm{ne}[{\color{purple}1}] = \{{\color{green}4}, {\color{blue}5}, {\color{red}3}\}
$$

$$
\mathrm{agg}[n = 1, k = 1] = \mathbf{h}_1^{(4)} + \mathbf{h}_1^{(5)} + \mathbf{h}_1^{(3)}
$$

Then a linear transform to the current node vector and the aggregate for the
current node and add a bias.

$$
\mathbf{h}_{k+1}^{(n)} = \mathbf{a}\Big[\underset{D\times 1}{\boldsymbol{\beta}_k} + \underset{D\times D}{\boldsymbol{\Omega}_k} \cdot \underset{D\times 1}{\mathbf{h}_k^{(n)}} + \underset{D\times D}{\boldsymbol{\Omega}_k} \cdot \underset{D\times 1}{\mathrm{agg}[n, k]}\Big]
$$


## Graph convolution layers

![](figs/20-Graph-Neural-Networks/graph-convolution-layers.png)

**Input**


$$
\mathbf{h}_1^{(n)} = \mathbf{a}\Big[\boldsymbol{\beta}_0 + \boldsymbol{\Omega}_0 \mathbf{x}^{(n)} + \boldsymbol{\Omega}_0 \mathrm{agg}[n, 0]\Big]
$$


**1^st^ Layer**


$$
\mathbf{h}_{k+1}^{(n)} = \mathbf{a}\Big[\boldsymbol{\beta}_k + \boldsymbol{\Omega}_k \mathbf{h}_k^{(n)} + \boldsymbol{\Omega}_k \mathrm{agg}[n, k]\Big]
$$


***k*+1^st^ Layer**


## Example Graph Convolution Network (GCN) layer: matrix form

We apply the following equation

$$
\mathbf{h}_{k+1}^{(n)} = \mathbf{a}\Big[\beta_k + \Omega_k \cdot \mathbf{h}_k^{(n)} + \Omega_k \cdot \mathrm{agg}[n, k]\Big]
$$

to the entire node hidden layers matrix, $\mathbf{H}_k$, by noting that
$\mathbf{H}_k\mathbf{A}$ produces a matrix where the $n^{th}$ column is
$\mathrm{agg}[n, k]$.

$$
\begin{aligned}
\mathbf{H}_{k+1} &= \mathbf{a}\left[\boldsymbol{\beta}_k \mathbf{1}^T + \boldsymbol{\Omega}_k \mathbf{H}_k + \boldsymbol{\Omega}_k \mathbf{H}_k \mathbf{A}\right] \\
&= \mathbf{a}\left[\boldsymbol{\beta}_k \mathbf{1}^T + \boldsymbol{\Omega}_k \mathbf{H}_k (\mathbf{A} + \mathbf{I})\right],
\end{aligned}
$$

Note that this is (1) equivariant to permutations, (2) handles arbitrary number
of neighbors, (3) exploits graph structure and (4) share parameters


This layer in code, with a check of claim (1) on the four-node graph, node
data $\mathbf{X}$ and permutation $\mathbf{P}$ from the permutation slides. The
cell relabels the graph as $\mathbf{X}' = \mathbf{X}\mathbf{P}$ and
$\mathbf{A}' = \mathbf{P}^T\mathbf{A}\mathbf{P}$ (printing both, to compare with
the slide), runs the same layer on both versions, and checks that the output is
relabelled in the same way, $\mathbf{H}_1' = \mathbf{H}_1\mathbf{P}$, while a
mean-pooled graph-level output does not change at all.

In [ ]:
# Adapted from UDL notebook 13.2 (MIT)
import numpy as np

rng = np.random.default_rng(0)

X = np.array([[1, 2, 3, 4],
              [5, 6, 7, 8],
              [9, 10, 11, 12]], dtype=float)       # D x N, one column per node
A = np.array([[0, 1, 0, 0],
              [1, 0, 1, 1],
              [0, 1, 0, 1],
              [0, 1, 1, 0]])
P = np.array([[0, 0, 0, 1],
              [0, 0, 1, 0],
              [1, 0, 0, 0],
              [0, 1, 0, 0]])
D, N = X.shape
D_out = 3

def relu(z):
    return np.maximum(z, 0)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def gcn_layer(H, A, Omega, beta):
    """H_{k+1} = a[beta 1^T + Omega H (A + I)]"""
    ones = np.ones((1, H.shape[1]))
    return relu(beta @ ones + Omega @ H @ (A + np.eye(len(A))))

def graph_output(H, omega, beta):                  # mean pooling, then sigmoid
    return sigmoid(beta + omega @ H @ np.ones((H.shape[1], 1)) / H.shape[1]).item()

Omega0 = rng.normal(0, np.sqrt(2 / D), size=(D_out, D))
beta0 = rng.normal(0, 0.1, size=(D_out, 1))
omega1 = rng.normal(0, 0.1, size=(1, D_out))
beta1 = 0.0

X_p = X @ P
A_p = P.T @ A @ P
print("X' = X P =\n", X_p.astype(int))
print("A' = P^T A P =\n", A_p)

H1 = gcn_layer(X, A, Omega0, beta0)
H1_p = gcn_layer(X_p, A_p, Omega0, beta0)
print("\nH1 =\n", H1.round(2))
print("H1' =\n", H1_p.round(2))
print("equivariant, H1' = H1 P:", np.allclose(H1_p, H1 @ P))

print(f"\ngraph-level output, original labels: {graph_output(H1, omega1, beta1):.6f}")
print(f"graph-level output, permuted labels: {graph_output(H1_p, omega1, beta1):.6f}")

# Graph & Node classification

Stacking GCN layers and adding one of the heads from before gives complete
models. Graph classification trains on many small graphs; node classification
often works on one huge graph, which forces new ways of forming batches.


## Graph classification example

We can put it all together and add a sigmoid layer

$$
\begin{aligned}
\mathbf{H}_1 &= \mathbf{a}\left[\boldsymbol{\beta}_0 \mathbf{1}^T + \boldsymbol{\Omega}_0 \mathbf{X} (\mathbf{A} + \mathbf{I})\right] \\
\mathbf{H}_2 &= \mathbf{a}\left[\boldsymbol{\beta}_1 \mathbf{1}^T + \boldsymbol{\Omega}_1 \mathbf{H}_1 (\mathbf{A} + \mathbf{I})\right] \\
\vdots \;&=\; \vdots \\
\mathbf{H}_K &= \mathbf{a}\left[\boldsymbol{\beta}_{K-1} \mathbf{1}^T + \boldsymbol{\Omega}_{K-1} \mathbf{H}_{K-1} (\mathbf{A} + \mathbf{I})\right] \\
\mathrm{f}[\mathbf{X}, \mathbf{A}, \boldsymbol{\Phi}] &= \mathrm{sig}\big[\beta_K + \boldsymbol{\omega}_K \mathbf{H}_K \underbrace{\mathbf{1}/N}_{\text{Mean pooling}}\big],
\end{aligned}
$$


For classification on molecules,

$X \in \mathbb{R}^{118 \times N}$: one hot encoding of 118 elements

$\Omega_0 \in \mathbb{R}^{D \times 118}$: convert to $D$-dimensional embeddings

$\beta_K$: is a scalar

$\omega_K$: a $1 \times D$ parameters row vector


*See UDL notebook 13.2.*

## Inductive vs. Transductive

![](figs/20-Graph-Neural-Networks/inductive-vs-transductive.png)

[supervised learning]{.underline}: train with the labeled graphs and then run
inference on the unlabeled (test) graphs


[semi-supervised learning]{.underline}: train with the labeled nodes, then run
inference to determine label for unlabeled nodes


## Node classification example

Assume *transductive* binary node *classification* with
[millions of nodes]{.underline}, *partially labeled*.

Same network body as graph classification, but different head:

$$
\mathbf{f}[\mathbf{X}, \mathbf{A}, \boldsymbol{\Phi}] = \mathrm{sigmoid}[\beta_K \mathbf{1}^T + \boldsymbol{\omega}_K \mathbf{H}_K]
$$

No mean pooling. Output is $1 \times N$.

Train with binary cross-entropy loss on nodes with labels.


![](figs/20-Graph-Neural-Networks/node-classification-network.png)


The graph-classification network and this node-classification head side by
side, on the ethanol molecule of UDL notebook 13.2. Its nine atoms are one-hot
encoded over 118 elements ($\mathbf{X}$ is $118 \times 9$) and pass through
$K = 3$ GCN layers of width $D = 200$ with random weights. Two heads share that
body: the graph head mean-pools and gives one probability for the molecule, the
node head gives one probability per atom. The network is untrained, so the
probabilities mean nothing; the point is the shapes, and what each head does
when the atoms are relabelled.

In [ ]:
# Adapted from UDL notebook 13.2 (MIT)
import numpy as np

rng = np.random.default_rng(0)

atoms = ["H", "H", "C", "H", "H", "C", "H", "O", "H"]   # ethanol, nodes 0..8
bonds = [(0, 2), (1, 2), (3, 2), (2, 5), (4, 5), (6, 5), (7, 5), (8, 7)]
atomic_number = {"H": 1, "C": 6, "O": 8}
K, D, n_elements = 3, 200, 118

N = len(atoms)
A = np.zeros((N, N))
for m, n in bonds:
    A[m, n] = A[n, m] = 1
X = np.zeros((n_elements, N))
for n, atom in enumerate(atoms):
    X[atomic_number[atom] - 1, n] = 1              # one-hot column per atom

def relu(z):
    return np.maximum(z, 0)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

dims = [n_elements] + [D] * K
Omegas = [rng.normal(0, np.sqrt(2 / d_in), size=(D, d_in)) for d_in in dims[:-1]]
betas = [np.zeros((D, 1)) for _ in range(K)]
omega_K = rng.normal(0, np.sqrt(1 / D), size=(1, D))
beta_K = 0.0

def body(X, A):
    H = X
    for Omega, beta in zip(Omegas, betas):
        H = relu(beta @ np.ones((1, N)) + Omega @ H @ (A + np.eye(N)))
    return H                                       # H_K, D x N

def graph_head(H):                                 # sig[beta_K + omega_K H_K 1 / N]
    return sigmoid(beta_K + omega_K @ H @ np.ones((N, 1)) / N)

def node_head(H):                                  # sig[beta_K 1^T + omega_K H_K]
    return sigmoid(beta_K * np.ones((1, N)) + omega_K @ H)

H_K = body(X, A)
print("H_K shape:", H_K.shape)
print(f"graph head, shape {graph_head(H_K).shape}: Pr(y = 1) = {graph_head(H_K).item():.4f}")
print(f"node head, shape {node_head(H_K).shape}: Pr(y_n = 1) =", node_head(H_K).round(3).ravel())

P = np.eye(N)[:, rng.permutation(N)]               # random relabelling of the atoms
H_K_p = body(X @ P, P.T @ A @ P)
print("\nAfter relabelling the atoms:")
print("  graph head unchanged (invariant):", np.allclose(graph_head(H_K_p), graph_head(H_K)))
print("  node head permuted (equivariant):", np.allclose(node_head(H_K_p), node_head(H_K) @ P))

## Node classification example: challenges

Assume *transductive* binary node *classification* with
[millions of nodes]{.underline}, *partially labeled*.

Challenges:

1. memory limitations: need to store every node and hidden layer embedding during training
2. how to perform SGD with basically one batch!



## Solutions: Choosing batches for graphs

1. Choose random subset of nodes
2. Neighborhood sampling
3. Graph partitioning

## Batches: Random subset


![](figs/20-Graph-Neural-Networks/batch-receptive-field.png)


You can pick a random batch of labeled nodes at each training step,
and only include them and their "k-hop neighborhoods" in the GCN case.
Take brown node in layer 2 for example…

Each node is dependent on the same node in the previous layer and its neighbors
because of agg[].

$$
\mathbf{h}_{k+1}^{(n)} = \mathbf{a}\Big[\beta_k + \Omega_k \cdot \mathbf{h}_k^{(n)} + \Omega_k \cdot \mathrm{agg}[n, k]\Big]
$$


**With many layers and dense connection, it can quickly expand to encompass every node.**


## Neighborhood Sampling

![](figs/20-Graph-Neural-Networks/neighborhood-sampling.png)


**All neighbors (a):**

Use every neighbor

<br>

**Neighborhood Sampling (b):**

Use a random sample of at most $n$ of the neighbors.

Here $n = 3$.

*See UDL notebook 13.3*


## Graph Partitioning

![](figs/20-Graph-Neural-Networks/graph-partitioning.png)

Disconnect edges of the original to create maximally connected (maximum internal
edge density) disjoint subsets

Split into train, test and validation sets and train just like in the inductive
setting.

## Alternatives to the Neighbor Sum

- **Diagonal enhancement**: current node is multiplied by $(1 + \epsilon_k)$, where $\epsilon_k$ is a learned scalar for each layer

  $$
  \mathbf{H}_{k+1} = \mathbf{a}[\beta_k \mathbf{1}^T + \boldsymbol{\Omega}_k \mathbf{H}_k (\mathbf{A} + (1 + \epsilon_k)\mathbf{I})]
  $$

- **Residual connections**: add the current node to the output, after the activation

  $$
  \mathbf{H}_{k+1} = \mathbf{a}[\beta_k \mathbf{1}^T + \boldsymbol{\Omega}_k \mathbf{H}_k \mathbf{A}] + \mathbf{H}_k
  $$

- **Mean aggregation**: take average instead of sum of neighbors

  $$
  \mathrm{agg}[n] = \frac{1}{|\mathrm{ne}[n]|} \sum_{m \in \mathrm{ne}[n]} \mathbf{h}_m
  $$

- **Kipf normalization**: downweight neighboring nodes with a lot of neighbors

  $$
  \mathrm{agg}[n] = \sum_{m \in \mathrm{ne}[n]} \frac{h_m}{\sqrt{|\mathrm{ne}[n]||\mathrm{ne}[m]|}}
  $$

- **Max pool aggregation**: element-wise max of all neighbors to current node

  $$
  \mathrm{agg}[n] = \max_{m \in \mathrm{ne}[n]}[\mathbf{h}_m]
  $$

UDL book sections 13.8.1 – 13.8.5

Each aggregation above is $\mathbf{H}_k$ multiplied on the right by an
$N \times N$ matrix: $\mathbf{A}$ for the sum, $\mathbf{A}\mathbf{D}^{-1}$ for
the mean (column $n$ divided by the degree of node $n$), and
$\mathbf{D}^{-1/2}\mathbf{A}\mathbf{D}^{-1/2}$ for Kipf normalization, with
$\mathbf{D}$ the diagonal matrix of node degrees. The GCN of Kipf and Welling
adds the self-loops first and normalizes
$\tilde{\mathbf{D}}^{-1/2}(\mathbf{A} + \mathbf{I})\tilde{\mathbf{D}}^{-1/2}$,
with $\tilde{\mathbf{D}}$ the degrees of $\mathbf{A} + \mathbf{I}$. Column $n$
of each matrix holds the weights node $n$ gives to the nodes it aggregates. The
cell prints them for node 6 of the eight-node graph, whose neighbors 5, 7 and 8
have degrees 5, 1 and 2, and then aggregates a constant feature: the sum grows
with the degree, the mean does not, and Kipf normalization lies in between.

In [ ]:
import numpy as np
import pandas as pd

edges = [(1, 2), (1, 4), (2, 3), (2, 4), (2, 5), (3, 5), (4, 5),
         (5, 6), (5, 8), (6, 7), (6, 8)]
N = 8
node = 6

A = np.zeros((N, N))
for m, n in edges:
    A[m - 1, n - 1] = A[n - 1, m - 1] = 1
deg = A.sum(axis=0)
A_I = A + np.eye(N)
deg_I = A_I.sum(axis=0)

aggregators = {
    "sum: A": A,
    "mean: A D^-1": A / deg,                       # divides column n by deg[n]
    "Kipf: D^-1/2 A D^-1/2": A / np.sqrt(np.outer(deg, deg)),
    "GCN: D~^-1/2 (A+I) D~^-1/2": A_I / np.sqrt(np.outer(deg_I, deg_I)),
}

rows = [m for m in range(N) if A_I[m, node - 1]]
weights = pd.DataFrame(
    {name: M[rows, node - 1] for name, M in aggregators.items()},
    index=[f"node {m + 1} (degree {int(deg[m])})" for m in rows],
)
print(f"Weights node {node} gives each node when aggregating:")
print(weights.round(3).to_string())

H = np.ones((1, N))                                # every node carries the value 1
print("\nAggregate of a constant feature at nodes 1..8:")
for name, M in aggregators.items():
    print(f"  {name:28s}", (H @ M).round(2).ravel())

## Aggregation by Attention

Weights depend on data at the nodes.

Apply linear transform to current node:

$$
\mathbf{H}'_k = \beta_k \mathbf{1}^T + \boldsymbol{\Omega}_k \mathbf{H}_k
$$

Then the similarity $s_{mn}$ of each transformed node embedding $\mathbf{h}'_m$ to
the transformed node embedding $\mathbf{h}'_n$ is computed by concatenating the
pairs, taking a dot product with a column vector $\phi_k$ of learned parameters,
and applying an activation function:

$$
s_{mn} = \mathrm{a}\left[\phi_k^T \begin{bmatrix} \mathbf{h}'_m \\ \mathbf{h}'_n \end{bmatrix}\right]
$$

$$
\mathbf{H}_{k+1} = \mathbf{a}[\mathbf{H}'_k \cdot \mathrm{Softmask}[\mathbf{S}, \mathbf{A} + \mathbf{I}]]
$$

## Softmask[S, A+I]

The function Softmask[S, A+I]

- computes the attention values by applying softmax operation separately to each
  column of its first argument S,
- but only after setting values where the second argument A + I is zero to
  negative infinity, so they do not contribute.
- This ensures that the attention to non-neighboring nodes is zero.

## Graph Attention

![Regular graph convolution.](figs/20-Graph-Neural-Networks/gcn-block-diagram.png)


![Graph attention. The figure's $\mathrm{Softmax}[\mathbf{S} \odot (\mathbf{A}+\mathbf{I})]$ means $\mathrm{Softmask}[\mathbf{S}, \mathbf{A}+\mathbf{I}]$: non-neighbors are set to $-\infty$, not to zero.](figs/20-Graph-Neural-Networks/graph-attention-block-diagram.png)


*Similar* to Transformer Self Attention, *except*

- K, Q and V are all the same
- Different similarity measure
- Only attends to neighbors


![](figs/20-Graph-Neural-Networks/dot-product-self-attention-block.png)



# Edge graphs

Some tasks attach their data to the edges rather than the nodes. These are
handled by converting the graph so that edges become nodes.


## Edge Graphs

![](figs/20-Graph-Neural-Networks/edge-graph-transformation.png)

Handled by simple transformation from node graphs.

Then process as node graph.

Transform back to edge graph.

# Summary

## Notebooks

You're encouraged to explore the UDL notebooks:

- UDL notebook 13.1 Graph representation
- UDL notebook 13.2 Graph classification
- UDL notebook 13.3 Neighborhood Sampling
- UDL notebook 13.4 Graph Attention Networks